In [0]:
# ============================================================
# SILVER LAYER - DATA CLEANING & TRANSFORMATION
# ============================================================

# FUNCTION IMPORTS
from pyspark.sql.functions import (
    col,
    when,
    to_date,
    year,
    month,
    dayofmonth
)

# LOAD BRONZE
silver = spark.table("bronze_retail")

In [0]:
# SILVER VERIFICATION
records_bronze = silver.count()

print(f"Records received from Bronze: {records_bronze}")

Records received from Bronze: 541909


In [0]:
# REMOVE EXACT DUPLICATES
records_before = silver.count()

silver = silver.dropDuplicates()

records_after = silver.count()

duplicates_removed = records_before - records_after

print("STEP 1 - EXACT DUPLICATES")
print(f"Records before:       {records_before:,}")
print(f"Duplicates removed:   {duplicates_removed:,}")
print(f"Records after:        {records_after:,}")
print()



STEP 1 - EXACT DUPLICATES
Records before:       541,909
Duplicates removed:   5,268
Records after:        536,641



In [0]:
# REMOVE ZERO QUANTITY
records_before = silver.count()

zero_quantity_count = (
    silver
    .filter(col("Quantity") == 0)
    .count()
)

silver = silver.filter(col("Quantity") != 0)

records_after = silver.count()

print("STEP 2 - ZERO QUANTITIES")
print(f"Records before:       {records_before:,}")
print(f"Zero quantities:      {zero_quantity_count:,}")
print(f"Records removed:      {records_before - records_after:,}")
print(f"Records after:        {records_after:,}")
print()

STEP 2 - ZERO QUANTITIES
Records before:       536,641
Zero quantities:      0
Records removed:      0
Records after:        536,641



In [0]:
# REMOVE INVALID NEGATIVE PRICES
records_before = silver.count()

negative_price_count = (
    silver
    .filter(col("UnitPrice") < 0)
    .count()
)

silver = silver.filter(col("UnitPrice") >= 0)

records_after = silver.count()

print("STEP 3 - NEGATIVE PRICES")
print(f"Records before:       {records_before:,}")
print(f"Negative prices:      {negative_price_count:,}")
print(f"Records removed:      {records_before - records_after:,}")
print(f"Records after:        {records_after:,}")
print()

STEP 3 - NEGATIVE PRICES
Records before:       536,641
Negative prices:      2
Records removed:      2
Records after:        536,639



In [0]:
# PRESERVE RETURNS
return_count = (
    silver
    .filter(col("Quantity") < 0)
    .count()
)

print("STEP 4 - RETURNS")
print(f"Return transactions preserved: {return_count:,}")
print("Returns are NOT removed because they are required")
print("to calculate Returns and Net Revenue.")
print()

STEP 4 - RETURNS
Return transactions preserved: 10,587
Returns are NOT removed because they are required
to calculate Returns and Net Revenue.



In [0]:
# HANDLE NULL DESCRIPTIONS
null_description_count = (
    silver
    .filter(col("Description").isNull())
    .count()
)

silver = silver.withColumn(
    "Description",
    when(
        col("Description").isNull(),
        "Unknown Product"
    ).otherwise(col("Description"))
)

print("STEP 5 - NULL DESCRIPTIONS")
print(f"Null descriptions handled: {null_description_count:,}")
print()


STEP 5 - NULL DESCRIPTIONS
Null descriptions handled: 1,454



In [0]:
# CREATE TRANSACTION TYPE
silver = silver.withColumn(
    "TransactionType",
    when(col("Quantity") < 0, "Return")
    .otherwise("Sale")
)

In [0]:
# CREATE DATE ATTRIBUTES
silver = (
    silver
    .withColumn("Date", to_date(col("InvoiceDate")))
    .withColumn("Year", year(col("InvoiceDate")))
    .withColumn("Month", month(col("InvoiceDate")))
    .withColumn("Day", dayofmonth(col("InvoiceDate")))
)


In [0]:
# CALCULATE TOTAL
silver = silver.withColumn(
    "Total",
    col("Quantity") * col("UnitPrice")
)

In [0]:
# FINAL SILVER COUNT
silver_records = silver.count()

total_removed = records_bronze - silver_records

print("SILVER SUMMARY")

print(f"Bronze records:       {records_bronze:,}")
print(f"Silver records:       {silver_records:,}")
print(f"Total records removed:{total_removed:,}")
print()

print("Expected removals:")
print(f"Duplicates:         {duplicates_removed:,}")
print(f"Zero quantities:    {zero_quantity_count:,}")
print(f"Negative prices:    {negative_price_count:,}")
print()

print(
    f"Removal reconciliation: "
    f"{duplicates_removed + zero_quantity_count + negative_price_count:,}"
)

print()

SILVER SUMMARY
Bronze records:       541,909
Silver records:       536,639
Total records removed:5,270

Expected removals:
Duplicates:         5,268
Zero quantities:    0
Negative prices:    2

Removal reconciliation: 5,270



In [0]:
# SAVE TO SILVER
silver.write.format("delta").mode("overwrite").saveAsTable("silver_retail")

print("Silver table saved successfully.")

Silver table saved successfully.


In [0]:
# FINAL PREVIEW
silver_final = spark.table("silver_retail")

silver_final.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------------+----------+----+-----+---+------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TransactionType|      Date|Year|Month|Day|             Total|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------------+----------+----+-----+---+------------------+
|   536375|   85123A|WHITE HANGING HEA...|       6|2010-12-01 09:32:00|     2.55|   17850.0|United Kingdom|           Sale|2010-12-01|2010|   12|  1|15.299999999999999|
|   536389|    22196|SMALL HEART MEASU...|      24|2010-12-01 10:03:00|     0.85|   12431.0|     Australia|           Sale|2010-12-01|2010|   12|  1|              20.4|
|   536392|    22128|PARTY CONES CANDY...|      12|2010-12-01 10:29:00|     1.25|   13705.0|United Kingdom|           Sale|2010-12-01|2010|   12|  1|      